In [ ]:
# %% [CELL 1] Configuration  <- the only cell you edit (SMOKE, MODEL_NAME)
# =============================================================================
# CROSS-ENCODER 1 — train + held-out evaluation (GPU)
# Amazon ML Challenge 2026 — Business Entity Resolution
# =============================================================================
# A transformer reads BOTH records' raw text (name incl. legal form, address,
# numbers, native script + IndicTrans2 translation) and scores the pair. It is
# used only on pairs v3 is unsure about (v3 probability inside BAND), and its
# score is blended with v3's by a small logistic regression.
#
#   1. training pairs: every true match + hard negatives (top-scoring non-matches
#      from all blocking layers) for training-bucket entities (500-2999)
#   2. fine-tune the cross-encoder (early-stop slice: buckets 500-799)
#   3. score the held-out pairs in v3's unsure band (heldout_v3.parquet from the
#      diagnostics), learn the blend, tune thresholds per country
#   4. report held-out F0.5: v3 alone vs v3 + cross-encoder, same 2-fold
#      cross-fitted protocol (tuned on one half of entities, scored on the other)
# Apply it to test ONLY if step 4 shows a gain.
#
# INPUTS   competition dataset, ckpt_train zips, Phase B + Phase C train outputs,
#          translations_train.parquet, heldout_v3.parquet (diagnostics output),
#          stage2_models_v3 (for its config: thresholds / one-owner)
# SETTINGS GPU T4 x1, Internet ON (downloads the model), Persistence Files.
# RUN      SMOKE = True first (~5-8 min, tiny subset: checks the whole pipeline on
#          the GPU and prints the measured speed), then SMOKE = False.
# OUTPUT   /kaggle/working/ce_model/  (model + tokenizer + ce_config.json)
# =============================================================================

SMOKE = False
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"   # Apache-2.0, 118M
# stronger but ~2-3x slower: "xlm-roberta-base" (MIT, 278M)

DATA_ROOT = "/kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/student_resource/dataset"
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"

TRAIN_ENT = 25_000            # training entities per country (smoke: 500)
VAL_ENT = 3_000               # early-stopping entities per country (smoke: 200)
NEG_PER_ENT = 8               # hard negatives per entity
RAND_NEG = 2                  # random negatives per entity (diversity)
MAX_LEN = 128                 # tokens for the pair
BATCH, PRED_BATCH = 64, 256
EPOCHS = 2                    # smoke: 1
LR = 3e-5
BAND = (0.02, 0.99)           # v3 probabilities re-scored by the cross-encoder
Q_CHUNK = 50_000
VAL_BUCKETS, ES_START, ES_END = 500, 500, 800
SEED = 42



# %% [CELL 2] Imports and helpers
# ---------------------------------------------------------------------------
# Logging, timing, zip-aware input finder.
# ---------------------------------------------------------------------------
import gc
import hashlib
import json
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import lightgbm as lgb
from tqdm.auto import tqdm

try:
    from rapidfuzz.process import cpdist
except ImportError:
    print("installing/upgrading rapidfuzz (needs Internet ON) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "rapidfuzz"], check=True)
    try:
        from rapidfuzz.process import cpdist
    except ImportError:
        raise SystemExit("rapidfuzz was upgraded: restart the session, then rerun this cell.")
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler

DATA_ROOT, WORK, INPUT_ROOT = Path(DATA_ROOT), Path(WORK), Path(INPUT_ROOT)
T_START = time.time()
TIMES = {}                     # stage -> [seconds, scalable?]


def log(msg):
    print(f"  [{(time.time()-T_START)/60:5.1f} min] {msg}", flush=True)


def mem(tag=""):
    rss = psutil.Process().memory_info().rss / 1e9
    print(f"    [MEM] {tag:<28} RAM={rss:5.1f} GB  free={psutil.virtual_memory().available/1e9:5.1f} GB",
          flush=True)


def bar(it=None, **kw):
    kw.setdefault("mininterval", 10)
    return tqdm(it, **kw)


def timed(stage, t0, scalable=True):
    """Accumulate wall time per stage. scalable: True = grows with the number of
    pairs, False = fixed cost, None = done once and reused by the full run."""
    TIMES.setdefault(stage, [0.0, scalable])[0] += time.time() - t0


def hash_bucket(eid):
    """Deterministic 0-9999 bucket; identical to the EDA / Notebook 2 split."""
    return int(hashlib.md5(eid.encode()).hexdigest(), 16) % 10000


def locate(marker):
    """All files named `marker` under INPUT_ROOT, plus those inside any .zip under
    INPUT_ROOT. Each zip holding the marker is unpacked once into
    WORK/_unzipped/<zip name>/ (Kaggle may keep uploaded zips packed)."""
    root_ok = INPUT_ROOT.exists()
    hits = sorted(INPUT_ROOT.rglob(marker)) if root_ok else []
    for z in (sorted(INPUT_ROOT.rglob("*.zip")) if root_ok else []):
        dest = WORK / "_unzipped" / z.stem
        done = dest / ".unpacked"
        if not done.exists():
            with zipfile.ZipFile(z) as zf:
                if not any(n.endswith("/" + marker) or n == marker for n in zf.namelist()):
                    continue
                log(f"unpacking {z.name} -> {dest}")
                zf.extractall(dest)
            done.touch()
        hits += sorted(dest.rglob(marker))
    return hits


def project(full_factor, label):
    """Print measured stage times and their projection to the full run."""
    print(f"\n  RUNTIME PROJECTION ({label})")
    print(f"    {'stage':<30}{'smoke (s)':>11}{'full (min)':>12}")
    total = 0.0
    for stage, (sec, scalable) in TIMES.items():
        # scalable None = already done and reused by the full run
        full = 0.0 if scalable is None else sec * (full_factor if scalable else 1.0)
        total += full
        print(f"    {stage:<30}{sec:>11.1f}{full/60:>12.1f}")
    print(f"    {'TOTAL':<30}{'':>11}{total/60:>12.1f}")
    return total


# %% [CELL 3] Text cleaning (shared)
# ---------------------------------------------------------------------------
# load_src and cleaning.
# ---------------------------------------------------------------------------
COLS = ["entity_id", "business_name", "business_address", "country"]


def load_src(path):
    """Read a source TSV. sep='\t' is mandatory: fields contain commas."""
    return pd.read_csv(path, sep="\t", dtype=str, names=COLS, header=0, keep_default_na=False)


# ---- cleaning: copied verbatim from Notebook 2 so both stages agree ----------
LEGAL_RE = (r"\b(llc|l\.l\.c|inc|corp|corporation|ltd|limited|pvt|private|"
            r"llp|lp|pllc|sas|sarl|eurl|sasu|snc|company)\b")

ADDR_ABBR = [
    (r"\brd\b", "road"), (r"\bst\b", "street"), (r"\bdr\b", "drive"),
    (r"\bave\b", "avenue"), (r"\bblvd\b", "boulevard"), (r"\bln\b", "lane"),
    (r"\bct\b", "court"), (r"\bcir\b", "circle"), (r"\bhno\b", "house number"),
    (r"\bno\b", "number"), (r"\bopp\b", "opposite"), (r"\bnr\b", "near"),
    (r"\bdist\b", "district"), (r"\bflr\b", "floor"),
]


def clean_text(s):
    """Shared cleaning. Non-Latin scripts are dropped here; L3 reads raw text."""
    s = s.fillna("").astype(str).str.lower()
    s = (s.str.normalize("NFKD")
          .str.encode("ascii", errors="ignore")
          .str.decode("ascii"))
    s = s.str.replace(r"##\s*", "", regex=True)
    s = s.str.replace(r"\bwww\.", "", regex=True)
    s = s.str.replace(r"\.(com|net|org|co|in|fr|uk|biz|info|io)\b", "", regex=True)
    s = s.str.replace(r"^[\s\-\.\,\!\@\#\$\%\^\&\*\>\<\'\"/\\]+", "", regex=True)
    s = s.str.replace("&", " and ", regex=False)
    s = s.str.replace(r"\bn/?a\b", " ", regex=True)
    s = s.str.replace(r"[^\w\s]", " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()


def clean_name(s):
    """Name: shared cleaning + legal suffixes + repeated words (FH FH -> FH)."""
    s = clean_text(s)
    s = s.str.replace(LEGAL_RE, " ", regex=True)
    s = s.str.replace(r"\b(\w+)\s+\1\b", r"\1", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()


def clean_addr(s):
    """Address: shared cleaning + abbreviation expansion."""
    s = clean_text(s)
    for pat, rep in ADDR_ABBR:
        s = s.str.replace(pat, rep, regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()


# ---- extra text columns for Stage 2 -------------------------------------------
# Street-type words and French articles are skipped when forming the address key,
# so "15 Rue de la Hoxie" -> "15_hoxie" (not "15_rue"), matching "351 Hoxie Avenue"
# style keys. This matters most for France, which training never sees.
SKIP_AFTER_NUMBER = {
    "road", "street", "drive", "avenue", "boulevard", "lane", "court", "circle",
    "way", "highway", "rue", "impasse", "allee", "chemin", "place", "route", "quai",
    "cours", "square", "de", "du", "des", "la", "le", "les", "l", "d",
}


def addr_key_string(caddr):
    """House number + the first following token that isn't a street word/article.
    Returned as a space-joined, sorted string so rapidfuzz can compare two of them."""
    toks = caddr.split()
    keys = set()
    for i, a in enumerate(toks):
        if a.isdigit():
            a2 = a.lstrip("0")
            j = i + 1
            while j < len(toks) and toks[j] in SKIP_AFTER_NUMBER:
                j += 1
            if a2 and j < len(toks):
                b = toks[j].lstrip("0") if toks[j].isdigit() else toks[j]
                if b:
                    keys.add(a2 + "_" + b)
    return " ".join(sorted(keys))


def prepare_texts(df):
    """All per-record text columns Stage 2 needs, computed once."""
    out = pd.DataFrame({"entity_id": df["entity_id"].values})
    out["cname"] = clean_name(df["business_name"]).values
    out["caddr"] = clean_addr(df["business_address"]).values
    out["digits"] = out["caddr"].str.findall(r"\b\d+\b").str.join(" ")
    out["nospace"] = out["cname"].str.replace(" ", "", regex=False)
    out["akeys"] = [addr_key_string(a) for a in out["caddr"]]
    out["nonlatin"] = np.array([any(ord(ch) > 0x024F for ch in s)
                                for s in df["business_name"]], dtype=np.int8)
    return out


# %% [CELL 4] String features (shared)
# ---------------------------------------------------------------------------
# Needed by the v3 definitions.
# ---------------------------------------------------------------------------
# ---- feature definitions (shared with Step 2) --------------------------------
def group_features(q, S):
    """Scores relative to each S1's own candidate list (q must be sorted).
    Scale-free features like rank and gap-to-best transfer across countries."""
    starts = np.flatnonzero(np.r_[True, q[1:] != q[:-1]])
    sizes = np.diff(np.r_[starts, len(q)])
    cols = []
    for j in range(S.shape[1]):
        s = S[:, j].astype(np.float32)
        gmax = np.repeat(np.maximum.reduceat(s, starts), sizes)
        order = np.lexsort((-s, q))
        rank = np.empty(len(q), np.float32)
        rank[order] = np.arange(len(q)) - np.repeat(starts, sizes)
        cols += [s, gmax - s, rank]
    cols += [(S[:, :3] > 0).sum(1) + (S[:, 3] > 0), np.repeat(sizes, sizes)]
    return np.column_stack(cols).astype(np.float32)


SCORERS = [("cname", "ratio", fuzz.ratio), ("cname", "token_set", fuzz.token_set_ratio),
           ("cname", "token_sort", fuzz.token_sort_ratio), ("cname", "partial", fuzz.partial_ratio),
           ("cname", "jaro_winkler", JaroWinkler.normalized_similarity),
           ("nospace", "ratio", fuzz.ratio), ("caddr", "token_set", fuzz.token_set_ratio),
           ("caddr", "token_sort", fuzz.token_sort_ratio), ("caddr", "ratio", fuzz.ratio),
           ("digits", "token_set", fuzz.token_set_ratio), ("akeys", "token_set", fuzz.token_set_ratio)]
FLAG_FEATURES = ["a_blank", "q_blank", "p_blank", "nonlatin", "len_diff"]
PRE_FEATURES = [f"{s}_{k}" for s in ("l1", "l2", "l3", "ak") for k in ("score", "gap", "rank")] \
    + ["n_layers", "n_cands", "is_s3"]
RICH_FEATURES = PRE_FEATURES + ["pf", "pf_gap", "pf_rank"] \
    + [f"{col}_{nm}" for col, nm, _ in SCORERS] + FLAG_FEATURES


def string_features(qt, pt, q, p):
    """String similarities + flags for pairs (q[i], p[i]); qt/pt are the prepared
    text tables aligned to qids/pids. Chunked to bound memory."""
    Q = {c: qt[c].to_numpy() for c in ("cname", "caddr", "digits", "nospace", "akeys")}
    P = {c: pt[c].to_numpy() for c in ("cname", "caddr", "digits", "nospace", "akeys")}
    qlen, plen = qt["cname"].str.len().to_numpy(), pt["cname"].str.len().to_numpy()
    q_nl, p_nl = qt["nonlatin"].to_numpy(), pt["nonlatin"].to_numpy()
    out = np.empty((len(q), len(SCORERS) + len(FLAG_FEATURES)), dtype=np.float32)
    for i in bar(range(0, len(q), FEAT_CHUNK), desc="string features", unit="chunk"):
        qs, ps = q[i:i + FEAT_CHUNK], p[i:i + FEAT_CHUNK]
        n = len(qs)
        for k, (col, _, sc) in enumerate(SCORERS):
            out[i:i + n, k] = cpdist(Q[col][qs].tolist(), P[col][ps].tolist(),
                                     scorer=sc, workers=-1)
        j = len(SCORERS)
        out[i:i + n, j] = (Q["caddr"][qs] == "") | (P["caddr"][ps] == "")
        out[i:i + n, j + 1] = Q["cname"][qs] == ""
        out[i:i + n, j + 2] = P["cname"][ps] == ""
        out[i:i + n, j + 3] = q_nl[qs] | p_nl[ps]
        out[i:i + n, j + 4] = np.abs(qlen[qs] - plen[ps])
    return out


# %% [CELL 5] v2 features (shared)
# ---------------------------------------------------------------------------
# Installs indic_transliteration if missing.
# ---------------------------------------------------------------------------
# ---- v2: transliteration, legal form, phonetic skeleton (shared) -------------
import re as _re
import unicodedata as _ud
try:
    from indic_transliteration import sanscript as _sanscript
    from indic_transliteration.sanscript import transliterate as _translit
except ImportError:
    print("installing indic_transliteration (MIT licence; needs Internet ON) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "indic_transliteration"], check=True)
    from indic_transliteration import sanscript as _sanscript
    from indic_transliteration.sanscript import transliterate as _translit

_BLOCK_SCRIPT = {0x0900: _sanscript.DEVANAGARI, 0x0980: _sanscript.BENGALI,
                 0x0A00: _sanscript.GURMUKHI, 0x0A80: _sanscript.GUJARATI,
                 0x0B00: _sanscript.ORIYA, 0x0B80: _sanscript.TAMIL,
                 0x0C00: _sanscript.TELUGU, 0x0C80: _sanscript.KANNADA,
                 0x0D00: _sanscript.MALAYALAM}
_LEGAL_TOKENS = {"pvt", "private", "pr", "pra", "ltd", "limited", "li", "llp", "pllc", "llc",
                 "lp", "inc", "incorporated", "corp", "corporation", "co", "company", "sas",
                 "sasu", "sarl", "eurl", "snc", "sa", "public"}
_LEGAL_PREFIX = ("prai", "pirai", "limi", "elel")    # transliterated private / limited / LLP
_FORM_KEYS = (("LLP", {"llp"}), ("PLLC", {"pllc"}), ("LLC", {"llc"}), ("LP", {"lp"}),
              ("INC", {"inc", "incorporated"}), ("CORP", {"corp", "corporation"}),
              ("SASU", {"sasu"}), ("SAS", {"sas"}), ("SARL", {"sarl"}), ("EURL", {"eurl"}),
              ("SNC", {"snc"}), ("SA", {"sa"}))
_SKEL_FOLD = (("ph", "f"), ("th", "t"), ("sh", "s"), ("kh", "k"), ("gh", "g"), ("bh", "b"),
              ("dh", "d"), ("ch", "c"), ("ck", "k"), ("c", "k"), ("q", "k"), ("x", "ks"),
              ("z", "s"), ("w", "v"), ("j", "g"), ("y", ""), ("v", "b"))   # Bengali renders B as v


def latin_tokens(name):
    """Lowercase Latin tokens of a name. Indic-script words are transliterated
    (ITRANS), accents removed, dots dropped (L.L.C -> llc), other punctuation
    splits words (Housing-Network -> housing network)."""
    s = str(name)
    s = _re.sub(r"\.(com|net|org|co|in|fr|uk|biz|info|io)\b", " ", s, flags=_re.I)
    s = s.replace(".", "")
    fast = all(ord(ch) <= 0x024F for ch in s)
    if fast:
        s = _re.sub(r"[^\w\s]", " ", s)
    else:
        # Indic vowel signs / virama are not \w, so only replace real punctuation
        # and symbols (Unicode categories P* and S*).
        s = "".join(" " if _ud.category(ch)[0] in "PS" else ch for ch in s)
    out = []
    for w in s.split():
        indic = False
        if not fast:
            blk = next((b for b in _BLOCK_SCRIPT if any(b <= ord(c) < b + 0x80 for c in w)), None)
            if blk is not None:
                indic = True
                try:
                    w = _translit(w, _BLOCK_SCRIPT[blk], _sanscript.ITRANS)
                except Exception:
                    pass
                if blk == 0x0B80:                     # Tamil: library leaves some letters
                    w = w.replace("\u0BA9", "n").replace("\u0BB1", "r")
        w = _ud.normalize("NFKD", w).encode("ascii", "ignore").decode().lower()
        if indic and blk == 0x0B80:
            # Tamil has no aspirated/voiced contrast; the library writes ச க ப ட/த
            # as jh gh bh dh — fold them to s k p t.
            for a, b in (("jh", "s"), ("gh", "k"), ("bh", "p"), ("dh", "t")):
                w = w.replace(a, b)
        w = _re.sub(r"[^a-z0-9]", "", w)
        if indic:
            w = _re.sub(r"([bcdfghjklmnpqrstvwxyz])a$", r"\1", w)    # inherent final 'a'
        if w:
            out.append(w)
    return out


def legal_form(toks):
    """Canonical legal form: PVT (private limited), PLC, LTD, LLP, LLC, INC, ...
    Spelling variants map to one code (Pvt Ltd = Private Limited = प्रा. लि.)."""
    s = set(toks)
    has_pvt = bool(s & {"pvt", "private", "pr", "pra"}) or any(t.startswith(("prai", "pirai")) for t in toks)
    has_ltd = bool(s & {"ltd", "limited", "li"}) or any(t.startswith("limi") for t in toks)
    if has_pvt:
        return "PVT"
    if "public" in s and has_ltd:
        return "PLC"
    if any(t.startswith("elel") for t in toks):
        return "LLP"
    for code, keys in _FORM_KEYS:
        if s & keys:
            return code
    if has_ltd:
        return "LTD"
    if s & {"co", "company"}:
        return "CO"
    return ""


def skeleton(toks):
    """Phonetic consonant skeleton of the non-legal tokens: folds spelling variants,
    drops vowels, collapses repeats. 'teknolojis' and 'technologies' -> 'tknlgs'."""
    words = [t for t in toks if t not in _LEGAL_TOKENS and not t.startswith(_LEGAL_PREFIX)]
    s = " ".join(words)
    for a, b in _SKEL_FOLD:
        s = s.replace(a, b)
    s = _re.sub(r"[aeiou]", "", s)
    s = _re.sub(r"(.)\1+", r"\1", s).replace("m", "n")
    return _re.sub(r"\s+", " ", s).strip()


def prepare_extra(names):
    """Per-record legal form and phonetic skeleton."""
    forms, skels = [], []
    for nm in names:
        toks = latin_tokens(nm)
        forms.append(legal_form(toks))
        skels.append(skeleton(toks))
    return pd.DataFrame({"form": forms, "skel": skels})


EXTRA_FEATURES = ["form_same", "form_diff", "form_one_missing", "name_exact",
                  "skel_ratio", "skel_tset"]
RICH_V2 = RICH_FEATURES + EXTRA_FEATURES          # v1 features first, then v2 additions


def string_features_v2(qt, pt, q, p):
    """v1 string features + legal form relation, exact name, skeleton similarity.
    qt/pt must include the v2 columns 'form' and 'skel'."""
    base = string_features(qt, pt, q, p)
    fq, fp = qt["form"].to_numpy()[q], pt["form"].to_numpy()[p]
    cq, cp = qt["cname"].to_numpy()[q], pt["cname"].to_numpy()[p]
    sq, sp = qt["skel"].to_numpy()[q].tolist(), pt["skel"].to_numpy()[p].tolist()
    both = (fq != "") & (fp != "")
    extra = np.column_stack([
        both & (fq == fp),
        both & (fq != fp),
        (fq == "") != (fp == ""),
        (cq != "") & (cq == cp),
        cpdist(sq, sp, scorer=fuzz.ratio, workers=-1),
        cpdist(sq, sp, scorer=fuzz.token_set_ratio, workers=-1),
    ]).astype(np.float32)
    return np.column_stack([base, extra]).astype(np.float32)


def ensure_extra(prep_dir, ids_path, side, raw_names_by_id):
    """Cache <side>_x.parquet (form, skel) aligned to ids; compute if missing."""
    path = prep_dir / f"{side}_x.parquet"
    if path.exists():
        return
    ids = np.load(ids_path, allow_pickle=True)
    names = raw_names_by_id.reindex(ids)
    assert names.notna().all(), f"{path}: ids missing from the source files"
    t = time.time()
    x = prepare_extra(names.tolist())
    x.insert(0, "entity_id", ids)
    x.to_parquet(path, index=False)
    log(f"{path.parent.name}/{path.name}: {len(ids):,} rows in {time.time()-t:.0f}s")


def load_prepared(prep_dir, side):
    """v1 prepared text table + v2 extra columns, row-aligned."""
    t = pd.read_parquet(prep_dir / f"{side}.parquet")
    x = pd.read_parquet(prep_dir / f"{side}_x.parquet")
    assert (t["entity_id"].values == x["entity_id"].values).all(), "extra columns misaligned"
    t["form"], t["skel"] = x["form"].values, x["skel"].values
    return t


# %% [CELL 6] v3 shared code
# ---------------------------------------------------------------------------
# Candidate merge (all blocking layers).
# ---------------------------------------------------------------------------
# ---- v3: all blocking layers, translated names, fixed skeleton (Step 1/2 v3) ----
import os
from concurrent.futures import ProcessPoolExecutor as _PPE
import multiprocessing as _mp3

LAYERS = ["l1", "l2", "l3", "ak", "l5", "l6", "l7", "l8", "l8n", "l8a", "l2t"]
LAYER_SRC = {"old": ["l1", "l2", "l3", "ak"], "b": ["l5", "l6", "l7"], "c": ["l8", "l8n", "l8a", "l2t"]}
PRE_V3 = [f"{s}_{k}" for s in LAYERS for k in ("score", "gap", "rank")] + ["n_layers", "n_cands", "is_s3"]
RICH_V3 = (PRE_V3 + ["pf", "pf_gap", "pf_rank"] + [f"{col}_{nm}" for col, nm, _ in SCORERS]
           + FLAG_FEATURES + EXTRA_FEATURES + ["p_translated"])


def group_features_v3(q, S):
    """Per-query relative features for every layer score (q sorted)."""
    starts = np.flatnonzero(np.r_[True, q[1:] != q[:-1]])
    sizes = np.diff(np.r_[starts, len(q)])
    cols = []
    for j in range(S.shape[1]):
        s = S[:, j].astype(np.float32)
        gmax = np.repeat(np.maximum.reduceat(s, starts), sizes)
        order = np.lexsort((-s, q))
        rank = np.empty(len(q), np.float32)
        rank[order] = np.arange(len(q)) - np.repeat(starts, sizes)
        cols += [s, gmax - s, rank]
    cols += [(S > 0).sum(1), np.repeat(sizes, sizes)]
    return np.column_stack(cols).astype(np.float32)


def load_sources(paths):
    """paths: {"old": pairs.npz, "b": pairs_b.npz, "c": pairs_c.npz} -> arrays in memory."""
    D = {}
    for src, path in paths.items():
        z = np.load(path)
        D[src] = {k: z[k] for k in z.files}
    return D


def merge_chunk(D, q0, q1, npool):
    """Union of the candidate sources for queries q0..q1-1, sorted by (q, p);
    one column per layer score (0 when a source did not propose the pair)."""
    keys, parts = [], []
    for src, cols in LAYER_SRC.items():
        if src not in D:
            continue
        d = D[src]
        lo, hi = np.searchsorted(d["q"], [q0, q1])
        keys.append(d["q"][lo:hi].astype(np.int64) * npool + d["p"][lo:hi])
        parts.append((src, lo, hi))
    allk = np.concatenate(keys)
    uk, inv = np.unique(allk, return_inverse=True)
    out = {"q": (uk // npool).astype(np.int32), "p": (uk % npool).astype(np.int32),
           "S": np.zeros((len(uk), len(LAYERS)), dtype=np.float32)}
    has_label = all("label" in D[s] for s, _, _ in parts)
    if has_label:
        out["y"] = np.zeros(len(uk), dtype=np.int8)
    off = 0
    for src, lo, hi in parts:
        n = hi - lo
        idx = inv[off:off + n]                 # unique within a source: plain assignment
        for c in LAYER_SRC[src]:
            if c in D[src]:
                out["S"][idx, LAYERS.index(c)] = D[src][c][lo:hi]
        if has_label:
            out["y"][idx] = np.maximum(out["y"][idx], D[src]["label"][lo:hi])
        off += n
    return out


def skeleton3(toks):
    """Phonetic skeleton with the order fixed (m->n BEFORE collapsing repeats)."""
    words = [t for t in toks if t not in _LEGAL_TOKENS and not t.startswith(_LEGAL_PREFIX)]
    s = " ".join(words)
    for a, b in _SKEL_FOLD:
        s = s.replace(a, b)
    s = _re.sub(r"[aeiou]", "", s).replace("m", "n")
    s = _re.sub(r"(.)\1+", r"\1", s)
    return _re.sub(r"\s+", " ", s).strip()


def _x3_task(args):
    names, trans = args
    forms, skels = [], []
    for n, t in zip(names, trans):
        tr = latin_tokens(n)
        forms.append(legal_form(tr))           # form read from the original (translations may shorten it)
        skels.append(skeleton3(latin_tokens(t) if t else tr))
    return forms, skels


def ensure_x3(prep_dir, side, ids, names_by_id, trans):
    """Cache <side>_x3.parquet: legal form, fixed skeleton, and the cleaned name
    with translations substituted for non-Latin records."""
    path = prep_dir / f"{side}_x3.parquet"
    if path.exists():
        return
    t = time.time()
    names = names_by_id.reindex(ids)
    assert names.notna().all(), f"{path}: ids missing from the source files"
    names = names.tolist()
    tr = [trans.get(x, "") for x in ids]
    tasks = [(names[i:i + 100_000], tr[i:i + 100_000]) for i in range(0, len(names), 100_000)]
    forms, skels = [], []
    with _PPE(max_workers=os.cpu_count() or 2, mp_context=_mp3.get_context("fork")) as ex:
        for f_, s_ in ex.map(_x3_task, tasks):
            forms += f_; skels += s_
    base = pd.read_parquet(prep_dir / f"{side}.parquet", columns=["entity_id", "cname"])
    assert (base["entity_id"].values == ids).all(), "prepared texts misaligned"
    cname_t = base["cname"].to_numpy().copy()
    ti = np.array([i for i, x in enumerate(tr) if x], dtype=np.int64)
    if len(ti):
        cname_t[ti] = clean_name(pd.Series([tr[i] for i in ti])).to_numpy()
    x = pd.DataFrame({"entity_id": ids, "form": forms, "skel3": skels, "cname_t": cname_t})
    x["nospace_t"] = x["cname_t"].str.replace(" ", "", regex=False)
    x["translated"] = np.array([1 if v else 0 for v in tr], dtype=np.int8)
    x.to_parquet(path, index=False)
    log(f"{prep_dir.name}/{path.name}: {len(ids):,} rows ({int(x['translated'].sum()):,} translated) "
        f"in {time.time()-t:.0f}s")


def load_prepared_v3(prep_dir, side):
    """Prepared texts with v3 columns: translated cleaned names, fixed skeleton."""
    t = pd.read_parquet(prep_dir / f"{side}.parquet")
    x = pd.read_parquet(prep_dir / f"{side}_x3.parquet")
    assert (t["entity_id"].values == x["entity_id"].values).all(), "v3 columns misaligned"
    t["cname"], t["nospace"] = x["cname_t"].values, x["nospace_t"].values
    t["form"], t["skel"], t["translated"] = x["form"].values, x["skel3"].values, x["translated"].values
    return t


def string_features_v3(qt, pt, q, p):
    """v2 string features computed on translated names + fixed skeleton,
    plus a flag saying the candidate's name came from a translation."""
    base = string_features_v2(qt, pt, q, p)
    return np.column_stack([base, pt["translated"].to_numpy()[p].astype(np.float32)]).astype(np.float32)


# %% [CELL 7] Cross-encoder text + blend helpers (shared with CE-2 / apply)
# ---------------------------------------------------------------------------
# Pair text format and blend.
# ---------------------------------------------------------------------------
# ---- shared by CE-1 / CE-2 / apply: pair text + blend ----------------------------
def side_text(name, addr, trans=""):
    """One record as text: raw name (+ English translation) | raw address."""
    name, addr = str(name).strip(), str(addr).strip()
    if trans:
        name = f"{name} ({trans})"
    return f"{name} | {addr if addr else '[no address]'}"


def _logit(p):
    p = np.clip(np.asarray(p, dtype=np.float64), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def blend_features(p_v3, p_ce):
    z1, z2 = _logit(p_v3), _logit(p_ce)
    return np.column_stack([z1, z2, z1 * z2])


def blend_apply(coef, intercept, p_v3, p_ce):
    z = blend_features(p_v3, p_ce) @ np.asarray(coef) + intercept
    return 1.0 / (1.0 + np.exp(-z))


# %% [CELL 8] Cross-encoder model: training + prediction (GPU)
# ---------------------------------------------------------------------------
# Standard PyTorch loop, fp16 autocast.
# ---------------------------------------------------------------------------
# ---- cross-encoder: data, training, prediction (GPU) ---------------------------
import random
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

assert torch.cuda.is_available(), "GPU not enabled: set Accelerator to GPU T4"
DEV = "cuda"
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)


class _Pairs(Dataset):
    def __init__(self, enc, y):
        self.enc, self.y = enc, y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        d = {k: self.enc[k][i] for k in self.enc.keys()}
        d["labels"] = float(self.y[i])
        return d


def _collate(tok):
    def f(batch):
        labels = torch.tensor([b.pop("labels") for b in batch], dtype=torch.float32)
        out = tok.pad(batch, return_tensors="pt")
        out["labels"] = labels
        return out
    return f


@torch.no_grad()
def ce_predict(model, tok, A, B):
    """Match probability for text pairs (length-sorted batches for speed)."""
    model.eval()
    out = np.empty(len(A), dtype=np.float32)
    order = np.argsort([len(a) + len(b) for a, b in zip(A, B)])
    for i in bar(range(0, len(A), PRED_BATCH), desc="cross-encoder predict", unit="batch"):
        idx = order[i:i + PRED_BATCH]
        enc = tok([A[j] for j in idx], [B[j] for j in idx], truncation="longest_first",
                  max_length=MAX_LEN, padding=True, return_tensors="pt").to(DEV)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**enc).logits.squeeze(-1).float()
        out[idx] = torch.sigmoid(logits).cpu().numpy()
    return out


def ce_train(A, B, y, Av, Bv, yv):
    """Fine-tune; keep the epoch with the lowest early-stopping log-loss."""
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1).to(DEV)
    enc = tok(A, B, truncation="longest_first", max_length=MAX_LEN)
    dl = DataLoader(_Pairs(enc, y), batch_size=BATCH, shuffle=True, collate_fn=_collate(tok), num_workers=2)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * len(dl)
    sch = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    try:
        scaler = torch.amp.GradScaler("cuda")
    except Exception:
        scaler = torch.cuda.amp.GradScaler()
    lossf = torch.nn.BCEWithLogitsLoss()
    best_loss, best_state = float("inf"), None
    for ep in range(EPOCHS):
        model.train()
        t = time.time()
        for batch in bar(dl, desc=f"train epoch {ep + 1}/{EPOCHS}", unit="batch"):
            labels = batch.pop("labels").to(DEV)
            batch = {k: v.to(DEV) for k, v in batch.items()}
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(**batch).logits.squeeze(-1)
            loss = lossf(logits.float(), labels)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sch.step()
        pv = np.clip(ce_predict(model, tok, Av, Bv), 1e-6, 1 - 1e-6)
        vl = float(-np.mean(yv * np.log(pv) + (1 - yv) * np.log(1 - pv)))
        log(f"epoch {ep + 1}: {len(y)/(time.time()-t):.0f} pairs/s   early-stop log-loss {vl:.4f}")
        if vl < best_loss:
            best_loss = vl
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, tok


# %% [CELL 9] Inputs
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def by_country(marker, parent):
    return {f.parent.name: f.parent for f in locate(marker) if f.parent.parent.name == parent}

CDIR = {c: d for c, d in by_country("qids.npy", "ckpt_train").items() if (d / "pairs.npz").exists()}
BDIR, CCDIR = by_country("pairs_b.npz", "ckpt_b_train"), by_country("pairs_c.npz", "ckpt_c_train")
COUNTRIES = sorted(set(CDIR) & set(BDIR) & set(CCDIR))
tr_files = sorted(INPUT_ROOT.rglob("translations_train.parquet")) + sorted(WORK.glob("translations_train.parquet"))
ho_files = sorted(INPUT_ROOT.rglob("heldout_v3.parquet")) + sorted(WORK.glob("heldout_v3.parquet"))
cfg_files = [f for f in list(WORK.glob("stage2_models_v3/config.json")) + locate("config.json")
             if json.load(open(f)).get("version") == 3 and not json.load(open(f)).get("smoke")]
assert COUNTRIES, "train candidates missing: attach ckpt_train zips + Phase B / Phase C train outputs"
assert tr_files, "translations_train.parquet not found"
assert ho_files, "heldout_v3.parquet not found: attach the v3 diagnostics output"
assert cfg_files, "stage2_models_v3 (full run) not found: attach it (only its config.json is used)"
V3 = json.load(open(cfg_files[0]))
_t = pd.read_parquet(tr_files[0], columns=["entity_id", "translation"])
TRANS = dict(zip(_t["entity_id"], _t["translation"])); del _t
RECS = pd.concat([load_src(DATA_ROOT / "train" / f"train_source{i}.tsv") for i in (1, 2, 3)],
                 ignore_index=True).set_index("entity_id")
gt = pd.read_csv(DATA_ROOT / "train" / "train_ground_truth.tsv", sep="\t", dtype=str,
                 names=["s1", "m"], header=0, keep_default_na=False)
GT = {s: [x for x in m.split(",") if x] for s, m in zip(gt["s1"], gt["m"])}
del gt
if SMOKE:
    TRAIN_ENT, VAL_ENT, EPOCHS = 500, 200, 1
OUT = WORK / ("ce_model_smoke" if SMOKE else "ce_model")
print(f"CE-1 {'SMOKE' if SMOKE else 'FULL'} | model {MODEL_NAME} | countries {COUNTRIES} | "
      f"v3 held-out {V3['heldout_f05']:.4f}")


def texts(s1_ids, cand_ids):
    a = RECS.reindex(s1_ids); b = RECS.reindex(cand_ids)
    A = [side_text(n, ad) for n, ad in zip(a["business_name"], a["business_address"])]
    B = [side_text(n, ad, TRANS.get(e, "")) for e, n, ad in zip(cand_ids, b["business_name"], b["business_address"])]
    return A, B


# %% [CELL 10] Training pairs: all true matches + hard negatives
t0 = time.time()
rng = np.random.default_rng(SEED)
HARD = [LAYERS.index(c) for c in ("l8", "l2", "l1", "l3", "l5", "l7", "l2t")]
TR, VA = [], []
for c in COUNTRIES:
    qids = np.load(CDIR[c] / "qids.npy", allow_pickle=True)
    pids = np.load(CDIR[c] / "pids.npy", allow_pickle=True)
    qb = np.array([hash_bucket(x) for x in qids], dtype=np.int16)
    D = load_sources({"old": CDIR[c] / "pairs.npz", "b": BDIR[c] / "pairs_b.npz", "c": CCDIR[c] / "pairs_c.npz"})
    n_tr = n_va = 0
    for q0 in range(0, len(qids), Q_CHUNK):
        if n_tr >= TRAIN_ENT and n_va >= VAL_ENT:
            break
        m = merge_chunk(D, q0, min(len(qids), q0 + Q_CHUNK), len(pids))
        score = m["S"][:, HARD].sum(1) + m["S"][:, LAYERS.index("l8")]      # L8 counted twice
        df = pd.DataFrame({"q": m["q"], "p": m["p"], "y": m["y"], "s": score})
        b = qb[df["q"].to_numpy()]
        for want, dest, sel in ((TRAIN_ENT - n_tr, TR, (b >= ES_END) & (b < 3000)),
                                (VAL_ENT - n_va, VA, (b >= ES_START) & (b < ES_END))):
            if want <= 0:
                continue
            g = df[sel]
            ents = g["q"].unique()[:want]
            g = g[g["q"].isin(ents)]
            neg = g[g["y"] == 0].sort_values(["q", "s"], ascending=[True, False])
            hard = neg.groupby("q").head(NEG_PER_ENT)
            rest = neg.drop(hard.index)
            rnd = rest.groupby("q").sample(n=RAND_NEG, replace=True, random_state=SEED).drop_duplicates()
            pick = pd.concat([g[g["y"] == 1], hard, rnd])
            dest.append(pd.DataFrame({"s1": qids[pick["q"].to_numpy()], "cand": pids[pick["p"].to_numpy()],
                                      "y": pick["y"].to_numpy().astype(np.float32)}))
            if dest is TR:
                n_tr += len(ents)
            else:
                n_va += len(ents)
        del m, df
    del D
    gc.collect()
    log(f"[{c}] training entities {n_tr:,}, early-stop entities {n_va:,}")
TR = pd.concat(TR, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
VA = pd.concat(VA, ignore_index=True)
A_tr, B_tr = texts(TR["s1"], TR["cand"]); A_va, B_va = texts(VA["s1"], VA["cand"])
log(f"training pairs {len(TR):,} ({TR['y'].mean()*100:.1f}% true), early-stop pairs {len(VA):,}")
print("  example:", A_tr[0], "  <->  ", B_tr[0], "  label", TR["y"].iloc[0])
timed("training pairs", t0)


# %% [CELL 11] Fine-tune the cross-encoder
t0 = time.time()
model, tok = ce_train(A_tr, B_tr, TR["y"].to_numpy(), A_va, B_va, VA["y"].to_numpy())
timed("fine-tuning", t0)
OUT.mkdir(parents=True, exist_ok=True)
model.save_pretrained(OUT)
tok.save_pretrained(OUT)
log(f"model saved to {OUT}")


# %% [CELL 12] Score the held-out pairs in v3's unsure band
t0 = time.time()
H = pd.read_parquet(ho_files[0])
H = H[H["country"].isin(COUNTRIES)].reset_index(drop=True)
band = ((H["prob"] >= BAND[0]) & (H["prob"] < BAND[1])).to_numpy()
A_h, B_h = texts(H.loc[band, "s1"], H.loc[band, "cand"])
H["ce"] = np.nan
H.loc[band, "ce"] = ce_predict(model, tok, A_h, B_h)
rate = band.sum() / max(time.time() - t0, 1e-9)
log(f"held-out: {len(H):,} kept pairs, {int(band.sum()):,} in the band {BAND} ({band.mean()*100:.1f}%) "
    f"-> scored at {rate:.0f} pairs/s")
timed("held-out scoring", t0)


# %% [CELL 13] Blend + thresholds, evaluated with 2-fold cross-fitting over entities
ENT = []
for c in COUNTRIES:
    qids = np.load(CDIR[c] / "qids.npy", allow_pickle=True)
    qb = np.array([hash_bucket(x) for x in qids], dtype=np.int16)
    ENT += [(c, s) for s in qids[qb < VAL_BUCKETS]]
ENT = pd.DataFrame(ENT, columns=["country", "s1"])
ENT["true_n"] = [len(GT.get(s, ())) for s in ENT["s1"]]
ENT["ent"] = np.arange(len(ENT))
H = H.drop(columns=[c for c in ("ent",) if c in H]).merge(ENT[["s1", "ent"]], on="s1", how="left")
band = ((H["prob"] >= BAND[0]) & (H["prob"] < BAND[1])).to_numpy()
tc, n_ent = ENT["true_n"].to_numpy(), len(ENT)
ent, y, pv3, cty = H["ent"].to_numpy(), H["y"].to_numpy(), H["prob"].to_numpy(), H["country"].to_numpy()
pce = H["ce"].fillna(0).to_numpy()
fold_e = np.array([hash_bucket(s + "#fold") for s in ENT["s1"]]) % 2
fold = fold_e[ent]
GRID = np.round(np.arange(0.05, 0.99, 0.01), 2)


def f05_vec(pc, tp, tc):
    P = np.where(pc > 0, tp / np.maximum(pc, 1), 0.0)
    R = np.where(tc > 0, tp / np.maximum(tc, 1), 0.0)
    f = np.where(P + R > 0, 1.25 * P * R / np.maximum(0.25 * P + R, 1e-12), 0.0)
    return np.where((tc == 0) & (pc == 0), 1.0, f)


def f_of(sel, ents):
    pc = np.bincount(ent[sel], minlength=n_ent)
    tp = np.bincount(ent[sel], weights=y[sel], minlength=n_ent)
    return float(f05_vec(pc, tp, tc)[ents].mean())


CAND = H["cand"].to_numpy()


def owner_mask(score):
    """One-owner rule: a pair survives only if it is the top-scoring claim on its
    S2/S3 record. With a threshold on top, (score >= t) & owner_mask(score) is
    exactly the rule applied after thresholding, so this is computed once."""
    return score >= pd.Series(score).groupby([cty, CAND]).transform("max").to_numpy()


def tune_thresholds(score, ents_mask):
    """Per-country thresholds maximising F0.5 on the given entities (one-owner applied)."""
    om, th = owner_mask(score), {}
    for c in COUNTRIES:
        ce_ = np.flatnonzero(ents_mask & (ENT["country"] == c).to_numpy())
        cm = (cty == c) & om
        th[c] = float(max(GRID, key=lambda t: f_of((score >= t) & cm, ce_)))
    return th


def select(score, th):
    return (score >= np.array([th[c] for c in cty])) & owner_mask(score)


def fit_blend(rows):
    lr = LogisticRegression(C=1.0, max_iter=1000)
    lr.fit(blend_features(pv3[rows], pce[rows]), y[rows])
    return lr.coef_[0], float(lr.intercept_[0])


res = {"v3 alone": [], "v3 + cross-encoder (blend)": [], "cross-encoder alone in the band": []}
for k in (0, 1):
    tune_e, test_e = fold_e == k, fold_e != k
    tr_rows = band & (fold == k)
    coef, icpt = fit_blend(tr_rows)
    s_blend = pv3.copy(); s_blend[band] = blend_apply(coef, icpt, pv3[band], pce[band])
    s_ce = pv3.copy(); s_ce[band] = pce[band]
    for name, s in (("v3 alone", pv3), ("v3 + cross-encoder (blend)", s_blend),
                    ("cross-encoder alone in the band", s_ce)):
        th = tune_thresholds(s, tune_e)
        res[name].append(f_of(select(s, th), np.flatnonzero(test_e)))
base = np.mean(res["v3 alone"])
print("\n" + "=" * 80)
print(f"HELD-OUT F0.5, 2-fold cross-fitted ({n_ent:,} entities; band {BAND}: {int(band.sum()):,} pairs)")
print("=" * 80)
for name, v in res.items():
    print(f"  {name:<36} {np.mean(v):.4f}   ({(np.mean(v) - base)*100:+.2f} pts vs v3 alone)")
print(f"  AUC in the band: v3 {roc_auc_score(y[band], pv3[band]):.4f} | cross-encoder "
      f"{roc_auc_score(y[band], pce[band]):.4f}")
MIN_GAIN = 0.0005     # require >= +0.05 pts: smaller held-out differences are within noise
USE_CE = np.mean(res["v3 + cross-encoder (blend)"]) >= base + MIN_GAIN


# %% [CELL 14] Final blend on all held-out entities + save config
coef, icpt = fit_blend(band)
s_final = pv3.copy(); s_final[band] = blend_apply(coef, icpt, pv3[band], pce[band])
all_e = np.ones(n_ent, bool)
th = tune_thresholds(s_final, all_e)
_om = owner_mask(s_final)
th_one = float(max(GRID, key=lambda t: f_of((s_final >= t) & _om, np.flatnonzero(all_e))))
ce_cfg = {"model_name": MODEL_NAME, "max_len": MAX_LEN, "band": list(BAND),
          "blend_coef": [float(x) for x in coef], "blend_intercept": icpt,
          "thresholds": th, "threshold_unseen": th_one, "one_owner": True,
          "heldout_f05_v3": float(base), "heldout_f05_blend": float(np.mean(res["v3 + cross-encoder (blend)"])),
          "use_ce": bool(USE_CE), "smoke": SMOKE}
json.dump(ce_cfg, open(OUT / "ce_config.json", "w"), indent=2)
print(f"\n  thresholds {th} (unseen {th_one:.2f}) | blend coef {np.round(coef, 3).tolist()} "
      f"intercept {icpt:.3f}")
print(f"  RECOMMENDATION: {'APPLY the cross-encoder to test' if USE_CE else 'do NOT apply: held-out gain below +0.05 pts'}")
for f in sorted(OUT.iterdir()):
    print(f"  {OUT.name}/{f.name:<28} {f.stat().st_size/1e6:8.1f} MB")
if SMOKE:
    print("\n  Smoke run passed (scores above are meaningless at this size). Set SMOKE = False and rerun.")
log("CE-1 COMPLETE")

In [ ]:
# ---- stacker features (IDENTICAL in CE-1b and apply-v2) -------------------------
STACK_FEATURES = ["z_v3", "z_ce", "ce_rank", "ce_gap", "n_band", "v3_rank", "v3_gap",
                  "ent_n_conf", "is_s3", "cand_blank"]


def _logit_(p):
    p = np.clip(np.asarray(p, dtype=np.float64), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def stack_features(kept, band_mask, ce, cand_blank):
    """kept: DataFrame of ALL kept pairs with columns ent, cand, prob (v3).
    band_mask: bool array over kept rows (rows re-scored by the cross-encoder).
    ce: cross-encoder probability for the band rows (same order).
    cand_blank: 1 if the candidate's address is blank, for the band rows.
    Returns the feature matrix for the band rows."""
    k = kept[["ent", "prob"]].copy()
    g = k.groupby("ent")["prob"]
    k["v3_rank"] = g.rank(ascending=False, method="first")
    k["v3_gap"] = g.transform("max") - k["prob"]
    k["ent_n_conf"] = (k["prob"] >= 0.5).groupby(k["ent"]).transform("sum")
    b = k[band_mask].copy()
    b["ce"] = ce
    gb = b.groupby("ent")["ce"]
    b["ce_rank"] = gb.rank(ascending=False, method="first")
    b["ce_gap"] = gb.transform("max") - b["ce"]
    b["n_band"] = gb.transform("size")
    b["z_v3"], b["z_ce"] = _logit_(b["prob"]), _logit_(b["ce"])
    b["is_s3"] = kept.loc[band_mask, "cand"].astype(str).str.startswith("S3-").to_numpy().astype(np.float32)
    b["cand_blank"] = np.asarray(cand_blank, dtype=np.float32)
    return b[STACK_FEATURES].to_numpy(dtype=np.float32)


STACK_PARAMS = {"objective": "binary", "num_leaves": 31, "learning_rate": 0.05, "min_data_in_leaf": 50,
                "feature_fraction": 0.9, "bagging_fraction": 0.8, "bagging_freq": 1, "seed": 42, "verbose": -1}
STACK_ROUNDS = 400


# =============================================================================
# CE-1b — improve the blend + test a wider band (run in the CE-1 notebook's SAME
# session, after CE-1 finished: reuses the trained model and the held-out data)
# =============================================================================
# Compares, with the same 2-fold cross-fitting over entities as CE-1:
#   logistic blend (CE-1)  vs  LightGBM stacker with entity-context features
# for bands (0.02, 0.99) [current], (0.01, 0.995), (0.005, 0.999).
# Saves the best as ce_config_v2.json (+ stacker.txt) next to the model.
# RUNTIME ~10-20 min (mostly scoring the extra held-out pairs of the wider bands).
# =============================================================================
BANDS = [(0.02, 0.99), (0.01, 0.995), (0.005, 0.999)]
MIN_GAIN = 0.0005

# 1. cross-encoder scores for every held-out pair in the widest band
t0 = time.time()
lo_w, hi_w = BANDS[-1]
need = ((H["prob"] >= lo_w) & (H["prob"] < hi_w) & H["ce"].isna()).to_numpy()
if need.any():
    A_x, B_x = texts(H.loc[need, "s1"], H.loc[need, "cand"])
    H.loc[need, "ce"] = ce_predict(model, tok, A_x, B_x)
log(f"extra held-out pairs scored for the wider bands: {int(need.sum()):,} ({time.time()-t0:.0f}s)")
CAND_BLANK = (RECS["business_address"].reindex(H["cand"]).fillna("").str.strip() == "").to_numpy()
pce_all = H["ce"].to_numpy()


def evaluate(bnd, method):
    """Cross-fitted F0.5 for a band + combination method; also returns per-fold models."""
    bm = ((H["prob"] >= bnd[0]) & (H["prob"] < bnd[1])).to_numpy()
    Xs = stack_features(H[["ent", "cand", "prob"]], bm, pce_all[bm], CAND_BLANK[bm])
    yb = y[bm]
    fb = fold[bm]
    scores = []
    for k in (0, 1):
        s = pv3.copy()
        if method == "logistic":
            lr = LogisticRegression(C=1.0, max_iter=1000).fit(blend_features(pv3[bm][fb == k], pce_all[bm][fb == k]), yb[fb == k])
            s[np.flatnonzero(bm)] = lr.predict_proba(blend_features(pv3[bm], pce_all[bm]))[:, 1]
        else:
            st = lgb.train(STACK_PARAMS, lgb.Dataset(Xs[fb == k], yb[fb == k]), STACK_ROUNDS)
            s[np.flatnonzero(bm)] = st.predict(Xs)
        th = tune_thresholds(s, fold_e == k)
        scores.append(f_of(select(s, th), np.flatnonzero(fold_e != k)))
    return float(np.mean(scores)), int(bm.sum())


print("\n" + "=" * 84)
print("CE-1b: held-out F0.5, 2-fold cross-fitted (same protocol as CE-1)")
print("=" * 84)
print(f"  {'v3 alone':<44}{base:.4f}")
RES_B = {}
for bnd in BANDS:
    for method in ("logistic", "stacker"):
        f_, nb_ = evaluate(bnd, method)
        RES_B[(bnd, method)] = f_
        print(f"  band {str(bnd):<14} {method:<10} ({nb_:>9,} pairs)  {f_:.4f}   ({(f_ - base)*100:+.2f} pts vs v3)")
best_key = max(RES_B, key=RES_B.get)
cur = RES_B[(BANDS[0], "logistic")]
if RES_B[best_key] < cur + MIN_GAIN:
    best_key = (BANDS[0], "logistic")
BEST_BAND, BEST_METHOD = best_key
print(f"\n  BEST: band {BEST_BAND}, {BEST_METHOD}: {RES_B[best_key]:.4f} "
      f"({(RES_B[best_key] - cur)*100:+.2f} pts vs CE-1's logistic blend)")

# 2. final model on ALL held-out entities + thresholds, saved for apply-v2
bm = ((H["prob"] >= BEST_BAND[0]) & (H["prob"] < BEST_BAND[1])).to_numpy()
s = pv3.copy()
cfg2 = {"model_name": MODEL_NAME, "max_len": MAX_LEN, "band": list(BEST_BAND), "method": BEST_METHOD,
        "heldout_f05_v3": float(base), "heldout_f05": float(RES_B[best_key]), "smoke": SMOKE,
        "stack_features": STACK_FEATURES}
if BEST_METHOD == "logistic":
    lr = LogisticRegression(C=1.0, max_iter=1000).fit(blend_features(pv3[bm], pce_all[bm]), y[bm])
    s[np.flatnonzero(bm)] = lr.predict_proba(blend_features(pv3[bm], pce_all[bm]))[:, 1]
    cfg2.update(blend_coef=[float(x) for x in lr.coef_[0]], blend_intercept=float(lr.intercept_[0]))
else:
    Xs = stack_features(H[["ent", "cand", "prob"]], bm, pce_all[bm], CAND_BLANK[bm])
    st = lgb.train(STACK_PARAMS, lgb.Dataset(Xs, y[bm]), STACK_ROUNDS)
    s[np.flatnonzero(bm)] = st.predict(Xs)
    st.save_model(str(OUT / "stacker.txt"))
th2 = tune_thresholds(s, np.ones(n_ent, bool))
_om = owner_mask(s)
cfg2["thresholds"] = th2
cfg2["threshold_unseen"] = float(max(GRID, key=lambda t: f_of((s >= t) & _om, np.arange(n_ent))))
json.dump(cfg2, open(OUT / "ce_config_v2.json", "w"), indent=2)
print(f"  saved {OUT / 'ce_config_v2.json'}" + (" + stacker.txt" if BEST_METHOD == "stacker" else "")
      + f"   thresholds {th2} (unseen {cfg2['threshold_unseen']:.2f})")
print(f"  -> set BAND = {BEST_BAND} in the export cell before exporting the test pairs")
log("CE-1b COMPLETE")

In [ ]:
# Re-save ce_config_v2.json for band (0.02, 0.99) + stacker (matches the India/US clamp)
BEST_BAND, BEST_METHOD = (0.02, 0.99), "stacker"
bm = ((H["prob"] >= BEST_BAND[0]) & (H["prob"] < BEST_BAND[1])).to_numpy()
Xs = stack_features(H[["ent", "cand", "prob"]], bm, pce_all[bm], CAND_BLANK[bm])
st = lgb.train(STACK_PARAMS, lgb.Dataset(Xs, y[bm]), STACK_ROUNDS)
s = pv3.copy(); s[np.flatnonzero(bm)] = st.predict(Xs)
st.save_model(str(OUT / "stacker.txt"))
th2 = tune_thresholds(s, np.ones(n_ent, bool))
_om = owner_mask(s)
cfg2 = {"model_name": MODEL_NAME, "max_len": MAX_LEN, "band": list(BEST_BAND), "method": BEST_METHOD,
        "heldout_f05_v3": float(base), "heldout_f05": float(RES_B[(BEST_BAND, "stacker")]), "smoke": SMOKE,
        "stack_features": STACK_FEATURES, "thresholds": th2,
        "threshold_unseen": float(max(GRID, key=lambda t: f_of((s >= t) & _om, np.arange(n_ent))))}
json.dump(cfg2, open(OUT / "ce_config_v2.json", "w"), indent=2)
print("saved", OUT / "ce_config_v2.json", "| band", cfg2["band"], "| held-out", round(cfg2["heldout_f05"], 4),
      "| thresholds", th2, "| unseen", round(cfg2["threshold_unseen"], 2))

In [ ]:
# %% [CELL 1] Configuration
# =============================================================================
# CROSS-ENCODER 2 — score the test pairs in v3's unsure band (GPU)
# Amazon ML Challenge 2026 — Business Entity Resolution
# =============================================================================
# INPUTS   competition dataset, CE-1 output (ce_model/ with ce_config.json),
#          ce_band_test.parquet (export cell), translations_test.parquet
# SETTINGS GPU T4 x1, Internet ON, Persistence Files.
# OUTPUT   /kaggle/working/ce_test.parquet  (country, s1, cand, ce)
# RUN      SMOKE = True first (scores 20,000 pairs, prints speed + projection),
#          then SMOKE = False.
# =============================================================================

SMOKE = False
SMOKE_PAIRS = 20_000

DATA_ROOT = "/kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/student_resource/dataset"
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"
PRED_BATCH = 256
SEED = 42



# %% [CELL 2] Imports and helpers
# ---------------------------------------------------------------------------
# Logging, timing, zip-aware input finder.
# ---------------------------------------------------------------------------
import gc
import hashlib
import json
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import psutil
import lightgbm as lgb
from tqdm.auto import tqdm

try:
    from rapidfuzz.process import cpdist
except ImportError:
    print("installing/upgrading rapidfuzz (needs Internet ON) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "rapidfuzz"], check=True)
    try:
        from rapidfuzz.process import cpdist
    except ImportError:
        raise SystemExit("rapidfuzz was upgraded: restart the session, then rerun this cell.")
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler

DATA_ROOT, WORK, INPUT_ROOT = Path(DATA_ROOT), Path(WORK), Path(INPUT_ROOT)
T_START = time.time()
TIMES = {}                     # stage -> [seconds, scalable?]


def log(msg):
    print(f"  [{(time.time()-T_START)/60:5.1f} min] {msg}", flush=True)


def mem(tag=""):
    rss = psutil.Process().memory_info().rss / 1e9
    print(f"    [MEM] {tag:<28} RAM={rss:5.1f} GB  free={psutil.virtual_memory().available/1e9:5.1f} GB",
          flush=True)


def bar(it=None, **kw):
    kw.setdefault("mininterval", 10)
    return tqdm(it, **kw)


def timed(stage, t0, scalable=True):
    """Accumulate wall time per stage. scalable: True = grows with the number of
    pairs, False = fixed cost, None = done once and reused by the full run."""
    TIMES.setdefault(stage, [0.0, scalable])[0] += time.time() - t0


def hash_bucket(eid):
    """Deterministic 0-9999 bucket; identical to the EDA / Notebook 2 split."""
    return int(hashlib.md5(eid.encode()).hexdigest(), 16) % 10000


def locate(marker):
    """All files named `marker` under INPUT_ROOT, plus those inside any .zip under
    INPUT_ROOT. Each zip holding the marker is unpacked once into
    WORK/_unzipped/<zip name>/ (Kaggle may keep uploaded zips packed)."""
    root_ok = INPUT_ROOT.exists()
    hits = sorted(INPUT_ROOT.rglob(marker)) if root_ok else []
    for z in (sorted(INPUT_ROOT.rglob("*.zip")) if root_ok else []):
        dest = WORK / "_unzipped" / z.stem
        done = dest / ".unpacked"
        if not done.exists():
            with zipfile.ZipFile(z) as zf:
                if not any(n.endswith("/" + marker) or n == marker for n in zf.namelist()):
                    continue
                log(f"unpacking {z.name} -> {dest}")
                zf.extractall(dest)
            done.touch()
        hits += sorted(dest.rglob(marker))
    return hits


def project(full_factor, label):
    """Print measured stage times and their projection to the full run."""
    print(f"\n  RUNTIME PROJECTION ({label})")
    print(f"    {'stage':<30}{'smoke (s)':>11}{'full (min)':>12}")
    total = 0.0
    for stage, (sec, scalable) in TIMES.items():
        # scalable None = already done and reused by the full run
        full = 0.0 if scalable is None else sec * (full_factor if scalable else 1.0)
        total += full
        print(f"    {stage:<30}{sec:>11.1f}{full/60:>12.1f}")
    print(f"    {'TOTAL':<30}{'':>11}{total/60:>12.1f}")
    return total


# %% [CELL 3] Text cleaning (shared)
# ---------------------------------------------------------------------------
# load_src.
# ---------------------------------------------------------------------------
COLS = ["entity_id", "business_name", "business_address", "country"]


def load_src(path):
    """Read a source TSV. sep='\t' is mandatory: fields contain commas."""
    return pd.read_csv(path, sep="\t", dtype=str, names=COLS, header=0, keep_default_na=False)


# ---- cleaning: copied verbatim from Notebook 2 so both stages agree ----------
LEGAL_RE = (r"\b(llc|l\.l\.c|inc|corp|corporation|ltd|limited|pvt|private|"
            r"llp|lp|pllc|sas|sarl|eurl|sasu|snc|company)\b")

ADDR_ABBR = [
    (r"\brd\b", "road"), (r"\bst\b", "street"), (r"\bdr\b", "drive"),
    (r"\bave\b", "avenue"), (r"\bblvd\b", "boulevard"), (r"\bln\b", "lane"),
    (r"\bct\b", "court"), (r"\bcir\b", "circle"), (r"\bhno\b", "house number"),
    (r"\bno\b", "number"), (r"\bopp\b", "opposite"), (r"\bnr\b", "near"),
    (r"\bdist\b", "district"), (r"\bflr\b", "floor"),
]


def clean_text(s):
    """Shared cleaning. Non-Latin scripts are dropped here; L3 reads raw text."""
    s = s.fillna("").astype(str).str.lower()
    s = (s.str.normalize("NFKD")
          .str.encode("ascii", errors="ignore")
          .str.decode("ascii"))
    s = s.str.replace(r"##\s*", "", regex=True)
    s = s.str.replace(r"\bwww\.", "", regex=True)
    s = s.str.replace(r"\.(com|net|org|co|in|fr|uk|biz|info|io)\b", "", regex=True)
    s = s.str.replace(r"^[\s\-\.\,\!\@\#\$\%\^\&\*\>\<\'\"/\\]+", "", regex=True)
    s = s.str.replace("&", " and ", regex=False)
    s = s.str.replace(r"\bn/?a\b", " ", regex=True)
    s = s.str.replace(r"[^\w\s]", " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()


def clean_name(s):
    """Name: shared cleaning + legal suffixes + repeated words (FH FH -> FH)."""
    s = clean_text(s)
    s = s.str.replace(LEGAL_RE, " ", regex=True)
    s = s.str.replace(r"\b(\w+)\s+\1\b", r"\1", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()


def clean_addr(s):
    """Address: shared cleaning + abbreviation expansion."""
    s = clean_text(s)
    for pat, rep in ADDR_ABBR:
        s = s.str.replace(pat, rep, regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()


# ---- extra text columns for Stage 2 -------------------------------------------
# Street-type words and French articles are skipped when forming the address key,
# so "15 Rue de la Hoxie" -> "15_hoxie" (not "15_rue"), matching "351 Hoxie Avenue"
# style keys. This matters most for France, which training never sees.
SKIP_AFTER_NUMBER = {
    "road", "street", "drive", "avenue", "boulevard", "lane", "court", "circle",
    "way", "highway", "rue", "impasse", "allee", "chemin", "place", "route", "quai",
    "cours", "square", "de", "du", "des", "la", "le", "les", "l", "d",
}


def addr_key_string(caddr):
    """House number + the first following token that isn't a street word/article.
    Returned as a space-joined, sorted string so rapidfuzz can compare two of them."""
    toks = caddr.split()
    keys = set()
    for i, a in enumerate(toks):
        if a.isdigit():
            a2 = a.lstrip("0")
            j = i + 1
            while j < len(toks) and toks[j] in SKIP_AFTER_NUMBER:
                j += 1
            if a2 and j < len(toks):
                b = toks[j].lstrip("0") if toks[j].isdigit() else toks[j]
                if b:
                    keys.add(a2 + "_" + b)
    return " ".join(sorted(keys))


def prepare_texts(df):
    """All per-record text columns Stage 2 needs, computed once."""
    out = pd.DataFrame({"entity_id": df["entity_id"].values})
    out["cname"] = clean_name(df["business_name"]).values
    out["caddr"] = clean_addr(df["business_address"]).values
    out["digits"] = out["caddr"].str.findall(r"\b\d+\b").str.join(" ")
    out["nospace"] = out["cname"].str.replace(" ", "", regex=False)
    out["akeys"] = [addr_key_string(a) for a in out["caddr"]]
    out["nonlatin"] = np.array([any(ord(ch) > 0x024F for ch in s)
                                for s in df["business_name"]], dtype=np.int8)
    return out


# %% [CELL 4] Cross-encoder text + prediction
# ---------------------------------------------------------------------------
# Same text format and prediction code as CE-1.
# ---------------------------------------------------------------------------
# ---- shared by CE-1 / CE-2 / apply: pair text + blend ----------------------------
def side_text(name, addr, trans=""):
    """One record as text: raw name (+ English translation) | raw address."""
    name, addr = str(name).strip(), str(addr).strip()
    if trans:
        name = f"{name} ({trans})"
    return f"{name} | {addr if addr else '[no address]'}"


def _logit(p):
    p = np.clip(np.asarray(p, dtype=np.float64), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def blend_features(p_v3, p_ce):
    z1, z2 = _logit(p_v3), _logit(p_ce)
    return np.column_stack([z1, z2, z1 * z2])


def blend_apply(coef, intercept, p_v3, p_ce):
    z = blend_features(p_v3, p_ce) @ np.asarray(coef) + intercept
    return 1.0 / (1.0 + np.exp(-z))


MAX_LEN = 128   # replaced by ce_config.json's value in Cell 5


# ---- cross-encoder: data, training, prediction (GPU) ---------------------------
import random
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

assert torch.cuda.is_available(), "GPU not enabled: set Accelerator to GPU T4"
DEV = "cuda"
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)


class _Pairs(Dataset):
    def __init__(self, enc, y):
        self.enc, self.y = enc, y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        d = {k: self.enc[k][i] for k in self.enc.keys()}
        d["labels"] = float(self.y[i])
        return d


def _collate(tok):
    def f(batch):
        labels = torch.tensor([b.pop("labels") for b in batch], dtype=torch.float32)
        out = tok.pad(batch, return_tensors="pt")
        out["labels"] = labels
        return out
    return f


@torch.no_grad()
def ce_predict(model, tok, A, B):
    """Match probability for text pairs (length-sorted batches for speed)."""
    model.eval()
    out = np.empty(len(A), dtype=np.float32)
    order = np.argsort([len(a) + len(b) for a, b in zip(A, B)])
    for i in bar(range(0, len(A), PRED_BATCH), desc="cross-encoder predict", unit="batch"):
        idx = order[i:i + PRED_BATCH]
        enc = tok([A[j] for j in idx], [B[j] for j in idx], truncation="longest_first",
                  max_length=MAX_LEN, padding=True, return_tensors="pt").to(DEV)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**enc).logits.squeeze(-1).float()
        out[idx] = torch.sigmoid(logits).cpu().numpy()
    return out


def ce_train(A, B, y, Av, Bv, yv):
    """Fine-tune; keep the epoch with the lowest early-stopping log-loss."""
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1).to(DEV)
    enc = tok(A, B, truncation="longest_first", max_length=MAX_LEN)
    dl = DataLoader(_Pairs(enc, y), batch_size=BATCH, shuffle=True, collate_fn=_collate(tok), num_workers=2)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * len(dl)
    sch = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    try:
        scaler = torch.amp.GradScaler("cuda")
    except Exception:
        scaler = torch.cuda.amp.GradScaler()
    lossf = torch.nn.BCEWithLogitsLoss()
    best_loss, best_state = float("inf"), None
    for ep in range(EPOCHS):
        model.train()
        t = time.time()
        for batch in bar(dl, desc=f"train epoch {ep + 1}/{EPOCHS}", unit="batch"):
            labels = batch.pop("labels").to(DEV)
            batch = {k: v.to(DEV) for k, v in batch.items()}
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(**batch).logits.squeeze(-1)
            loss = lossf(logits.float(), labels)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sch.step()
        pv = np.clip(ce_predict(model, tok, Av, Bv), 1e-6, 1 - 1e-6)
        vl = float(-np.mean(yv * np.log(pv) + (1 - yv) * np.log(1 - pv)))
        log(f"epoch {ep + 1}: {len(y)/(time.time()-t):.0f} pairs/s   early-stop log-loss {vl:.4f}")
        if vl < best_loss:
            best_loss = vl
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, tok


# %% [CELL 5] Inputs + model
mfiles = [f for f in sorted(WORK.glob("ce_model/ce_config.json")) + locate("ce_config.json")
          if not json.load(open(f)).get("smoke")]          # same notebook as CE-1, or attached
assert mfiles, "CE-1 output (ce_model with ce_config.json, non-smoke) not found"
MDIR = mfiles[0].parent
CE_CFG = json.load(open(mfiles[0]))
MAX_LEN = CE_CFG["max_len"]
bfiles = sorted(INPUT_ROOT.rglob("ce_band_test.parquet")) + sorted(WORK.glob("ce_band_test.parquet"))
tfiles = sorted(INPUT_ROOT.rglob("translations_test.parquet")) + sorted(WORK.glob("translations_test.parquet"))
assert bfiles, "ce_band_test.parquet not found (export cell output)"
assert tfiles, "translations_test.parquet not found"
BT = pd.read_parquet(bfiles[0])
if SMOKE:
    BT = BT.sample(min(SMOKE_PAIRS, len(BT)), random_state=SEED).reset_index(drop=True)
_t = pd.read_parquet(tfiles[0], columns=["entity_id", "translation"])
TRANS = dict(zip(_t["entity_id"], _t["translation"])); del _t
RECS = pd.concat([load_src(DATA_ROOT / "test" / f"test_source{i}.tsv") for i in (1, 2, 3)],
                 ignore_index=True).set_index("entity_id")
tok = AutoTokenizer.from_pretrained(MDIR)
model = AutoModelForSequenceClassification.from_pretrained(MDIR).to(DEV)
print(f"CE-2 {'SMOKE' if SMOKE else 'FULL'} | model {MDIR} (held-out: v3 {CE_CFG['heldout_f05_v3']:.4f} -> "
      f"blend {CE_CFG['heldout_f05_blend']:.4f}; use_ce={CE_CFG['use_ce']}) | pairs {len(BT):,}")


# %% [CELL 6] Score
t0 = time.time()
a, b = RECS.reindex(BT["s1"]), RECS.reindex(BT["cand"])
A = [side_text(n, ad) for n, ad in zip(a["business_name"], a["business_address"])]
B = [side_text(n, ad, TRANS.get(e, "")) for e, n, ad in zip(BT["cand"], b["business_name"], b["business_address"])]
BT["ce"] = ce_predict(model, tok, A, B)
rate = len(BT) / max(time.time() - t0, 1e-9)
out = WORK / ("ce_test_smoke.parquet" if SMOKE else "ce_test.parquet")
BT[["country", "s1", "cand", "ce"]].to_parquet(out, index=False)
print(f"scored {len(BT):,} pairs at {rate:.0f} pairs/s -> {out}")
if SMOKE:
    full = len(pd.read_parquet(bfiles[0], columns=["s1"]))
    print(f"PROJECTION: {full:,} pairs at {rate:.0f} pairs/s = ~{full/rate/60:.0f} min. "
          f"Smoke passed: set SMOKE = False and rerun.")
log("CE-2 COMPLETE")

In [ ]:
# =============================================================================
# CE-1c — v4 + cross-encoder: refit the blend on v4's held-out probabilities
# Run in the CE-1 notebook. Best: the SAME session as CE-1 / CE-1b / CE-2 (reuses
# the model and the cross-encoder scores already computed). After a restart: rerun
# CE-1 Cells 1-9 (setup + inputs, ~5 min), then this cell; it reloads the model
# from /kaggle/working/ce_model and scores whatever it needs.
# INPUT  heldout_v4.parquet (B's export from the Step 1 v4 session)
# OUTPUT /kaggle/working/ce_model/ce_config_v4.json (+ stacker_v4.txt)
# =============================================================================
import numpy as np, pandas as pd, lightgbm as lgb, json
from sklearn.linear_model import LogisticRegression

BANDS_4 = [(0.02, 0.99), (0.01, 0.995)]
MIN_GAIN = 0.0005                       # v4 + cross-encoder must beat v4 alone by >= 0.05 pts
CE_DIR = WORK / "ce_model"

# ---- stacker features (IDENTICAL to CE-1b / apply) --------------------------------
STACK_FEATURES = ["z_v3", "z_ce", "ce_rank", "ce_gap", "n_band", "v3_rank", "v3_gap",
                  "ent_n_conf", "is_s3", "cand_blank"]     # "v3" = the base model's probability (here v4)
def _logit_(p):
    p = np.clip(np.asarray(p, dtype=np.float64), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))
def stack_features(kept, band_mask, ce, cand_blank):
    k = kept[["ent", "prob"]].copy()
    g = k.groupby("ent")["prob"]
    k["v3_rank"] = g.rank(ascending=False, method="first")
    k["v3_gap"] = g.transform("max") - k["prob"]
    k["ent_n_conf"] = (k["prob"] >= 0.5).groupby(k["ent"]).transform("sum")
    b = k[band_mask].copy()
    b["ce"] = ce
    gb = b.groupby("ent")["ce"]
    b["ce_rank"] = gb.rank(ascending=False, method="first")
    b["ce_gap"] = gb.transform("max") - b["ce"]
    b["n_band"] = gb.transform("size")
    b["z_v3"], b["z_ce"] = _logit_(b["prob"]), _logit_(b["ce"])
    b["is_s3"] = kept.loc[band_mask, "cand"].astype(str).str.startswith("S3-").to_numpy().astype(np.float32)
    b["cand_blank"] = np.asarray(cand_blank, dtype=np.float32)
    return b[STACK_FEATURES].to_numpy(dtype=np.float32)
STACK_PARAMS = {"objective": "binary", "num_leaves": 31, "learning_rate": 0.05, "min_data_in_leaf": 50,
                "feature_fraction": 0.9, "bagging_fraction": 0.8, "bagging_freq": 1, "seed": 42, "verbose": -1}
STACK_ROUNDS = 400

# ---- 1. v4 held-out predictions + the same held-out entities ----------------------
h4f = sorted(INPUT_ROOT.rglob("heldout_v4.parquet")) + sorted(WORK.glob("heldout_v4.parquet"))
assert h4f, "heldout_v4.parquet not found: attach B's export from the Step 1 v4 session"
H4 = pd.read_parquet(h4f[0])
H4 = H4[H4["country"].isin(COUNTRIES)].reset_index(drop=True)
if "ENT" not in globals():
    ENT = []
    for c in COUNTRIES:
        qids_ = np.load(CDIR[c] / "qids.npy", allow_pickle=True)
        qb_ = np.array([hash_bucket(x) for x in qids_], dtype=np.int16)
        ENT += [(c, s) for s in qids_[qb_ < VAL_BUCKETS]]
    ENT = pd.DataFrame(ENT, columns=["country", "s1"])
    ENT["true_n"] = [len(GT.get(s, ())) for s in ENT["s1"]]
    ENT["ent"] = np.arange(len(ENT))
H4 = H4.merge(ENT[["s1", "ent"]], on="s1", how="left")
assert H4["ent"].notna().all(), "heldout_v4 has entities outside the held-out split"
log(f"v4 held-out: {len(H4):,} kept pairs, {H4['s1'].nunique():,} of {len(ENT):,} held-out entities")

# ---- 2. cross-encoder scores: reuse what is known, score the rest ------------------
if "model" not in globals():
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tok = AutoTokenizer.from_pretrained(CE_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(CE_DIR).to(DEV)
lo_w, hi_w = min(b[0] for b in BANDS_4), max(b[1] for b in BANDS_4)
wide = ((H4["prob"] >= lo_w) & (H4["prob"] < hi_w)).to_numpy()
H4["ce"] = np.nan
if "H" in globals() and "ce" in H.columns:
    known = H.dropna(subset=["ce"]).drop_duplicates(["s1", "cand"]).set_index(["s1", "cand"])["ce"]
    H4.loc[wide, "ce"] = known.reindex(pd.MultiIndex.from_frame(H4.loc[wide, ["s1", "cand"]])).to_numpy()
need = wide & H4["ce"].isna().to_numpy()
log(f"band pairs {int(wide.sum()):,}: reused {int(wide.sum() - need.sum()):,}, scoring {int(need.sum()):,} new")
if need.any():
    A_, B_ = texts(H4.loc[need, "s1"], H4.loc[need, "cand"])
    H4.loc[need, "ce"] = ce_predict(model, tok, A_, B_)

# ---- 3. cross-fitted evaluation: v4 alone vs v4 + cross-encoder -------------------
ent4 = H4["ent"].to_numpy().astype(np.int64); y4 = H4["y"].to_numpy().astype(np.float64)
pv4 = H4["prob"].to_numpy().astype(np.float64); cty4 = H4["country"].to_numpy()
cand4 = H4["cand"].to_numpy(); pce4 = H4["ce"].to_numpy()
tc4, n4 = ENT["true_n"].to_numpy(), len(ENT)
fold_e4 = np.array([hash_bucket(s + "#fold") for s in ENT["s1"]]) % 2
fold4 = fold_e4[ent4]
blank4 = (RECS["business_address"].reindex(H4["cand"]).fillna("").str.strip() == "").to_numpy()
GRID4 = np.round(np.arange(0.05, 0.99, 0.01), 2)

def _f05(pc, tp, tc):
    P = np.where(pc > 0, tp / np.maximum(pc, 1), 0.0); R_ = np.where(tc > 0, tp / np.maximum(tc, 1), 0.0)
    f = np.where(P + R_ > 0, 1.25 * P * R_ / np.maximum(0.25 * P + R_, 1e-12), 0.0)
    return np.where((tc == 0) & (pc == 0), 1.0, f)
def _f_of(sel, ents):
    pc = np.bincount(ent4[sel], minlength=n4); tp = np.bincount(ent4[sel], weights=y4[sel], minlength=n4)
    return float(_f05(pc, tp, tc4)[ents].mean())
def _owner(score):          # strict one-owner, as applied on test
    return pd.Series(-score).groupby([cty4, cand4]).rank(method="first").to_numpy() == 1
def _tune(score, emask):
    om, th = _owner(score), {}
    for c in COUNTRIES:
        ce_ = np.flatnonzero(emask & (ENT["country"] == c).to_numpy()); cm = (cty4 == c) & om
        th[c] = float(max(GRID4, key=lambda t: _f_of((score >= t) & cm, ce_)))
    return th
def _select(score, th):
    return (score >= np.array([th[c] for c in cty4])) & _owner(score)
def _combined(bnd, method, fit_rows):
    bm = ((pv4 >= bnd[0]) & (pv4 < bnd[1])) & ~np.isnan(pce4)
    s = pv4.copy()
    if method == "logistic":
        lr = LogisticRegression(C=1.0, max_iter=1000).fit(blend_features(pv4[bm & fit_rows], pce4[bm & fit_rows]),
                                                          y4[bm & fit_rows])
        s[bm] = lr.predict_proba(blend_features(pv4[bm], pce4[bm]))[:, 1]
        return s, ("logistic", lr)
    X = stack_features(H4[["ent", "cand", "prob"]], bm, pce4[bm], blank4[bm])
    st = lgb.train(STACK_PARAMS, lgb.Dataset(X[fit_rows[bm]], y4[bm][fit_rows[bm]]), STACK_ROUNDS)
    s[bm] = st.predict(X)
    return s, ("stacker", st)
def _crossfit(bnd, method):
    out = []
    for k in (0, 1):
        if method == "none":
            s = pv4
        else:
            s, _ = _combined(bnd, method, fold4 == k)
        out.append(_f_of(_select(s, _tune(s, fold_e4 == k)), np.flatnonzero(fold_e4 != k)))
    return float(np.mean(out))

base4 = _crossfit(None, "none")
print("\n" + "=" * 84)
print(f"CE-1c: held-out F0.5, 2-fold cross-fitted ({n4:,} entities; strict one-owner)")
print("=" * 84)
print(f"  {'v4 alone':<44}{base4:.4f}")
RES4 = {}
for bnd in BANDS_4:
    for method in ("logistic", "stacker"):
        RES4[(bnd, method)] = _crossfit(bnd, method)
        print(f"  v4 + CE, band {str(bnd):<14} {method:<9} {RES4[(bnd, method)]:.4f}   "
              f"({(RES4[(bnd, method)] - base4)*100:+.2f} pts vs v4)")
best = max(RES4, key=RES4.get)
USE_CE4 = RES4[best] >= base4 + MIN_GAIN
print(f"\n  BEST: band {best[0]}, {best[1]}: {RES4[best]:.4f}   ->   "
      f"{'APPLY v4 + cross-encoder' if USE_CE4 else 'do NOT apply: submit v4 alone'}")

# ---- 4. final fit on all held-out entities + save --------------------------------
s_fin, (meth, mdl) = _combined(best[0], best[1], np.ones(len(H4), bool))
th4 = _tune(s_fin, np.ones(n4, bool)); om4 = _owner(s_fin)
cfg4 = {"base_model": "v4", "model_name": MODEL_NAME, "max_len": MAX_LEN, "band": list(best[0]), "method": meth,
        "stack_features": STACK_FEATURES, "thresholds": th4,
        "threshold_unseen": float(max(GRID4, key=lambda t: _f_of((s_fin >= t) & om4, np.arange(n4)))),
        "heldout_f05_base": base4, "heldout_f05": RES4[best], "use_ce": bool(USE_CE4), "smoke": False}
if meth == "logistic":
    cfg4.update(blend_coef=[float(x) for x in mdl.coef_[0]], blend_intercept=float(mdl.intercept_[0]))
else:
    mdl.save_model(str(CE_DIR / "stacker_v4.txt"))
json.dump(cfg4, open(CE_DIR / "ce_config_v4.json", "w"), indent=2)
print(f"  saved {CE_DIR / 'ce_config_v4.json'}" + (" + stacker_v4.txt" if meth == "stacker" else "")
      + f" | thresholds {th4} (unseen {cfg4['threshold_unseen']:.2f})")
log("CE-1c COMPLETE")

In [ ]:
!zip -r /kaggle/working/ce_model.zip /kaggle/working/ce_model/

In [ ]:
import shutil
from IPython.display import FileLink

# Compress /kaggle/working/ce_model into ce_model.zip
shutil.make_archive('ce_model', 'zip', '/kaggle/working/ce_model')

# Generate a direct download link
FileLink('ce_model.zip')